# Training a Tabular VAE

In this notebook you will train a **Variational Autoencoder (VAE)** on a table with both
**categorical** columns (e.g. education level) and **continuous** columns (e.g. income),
and then use it to **generate new synthetic rows**.

**The big picture**

```
table --preprocess--> numbers --encoder--> (mu, logvar) --sample--> z --decoder--> reconstruction
                                                                     |
                                                        sample z ~ N(0,1) --> NEW rows
```

**Files in this folder** (you do not need to edit them, but do read them!)

| File | Purpose |
|---|---|
| `encoders.py` | turns one continuous column into `m >= 1` numbers (and back) |
| `preprocessing.py` | one-hot encodes categoricals, applies the encoders, builds one matrix |
| `vae.py` | the VAE model and the loss function |

This notebook runs on the **CPU**, no GPU needed.

**Requirements:** `pip install torch numpy pandas scikit-learn matplotlib` (and `wandb` if you want experiment tracking).

## 0. Imports and settings

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

from encoders import StandardEncoder, ModeSpecificEncoder
from preprocessing import TabularPreprocessor
from vae import TabularVAE, vae_loss

# ----------------------- Settings you can play with -----------------------
EPOCHS = 100            # how many passes over the training data
BATCH_SIZE = 256        # rows per gradient update
LEARNING_RATE = 1e-3    # step size of the optimizer
HIDDEN_DIM = 64         # width of the hidden layers
LATENT_DIM = 8          # size of z
BETA = 1.0              # weight of the KL term in the loss
KL_WARMUP_EPOCHS = 10   # BETA grows slowly from ~0 to its full value during these epochs
VAL_FRACTION = 0.2      # share of the data held out for validation
SEED = 42

USE_WANDB = False       # set to True to log to Weights & Biases (needs `pip install wandb` and `wandb login`)
WANDB_PROJECT = "tabular-vae"
# --------------------------------------------------------------------------

torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Data

We create a small made-up dataset with some structure we can check for later:

* `income` grows with `age` and `education`
* `hours_per_week` has **two peaks** (part-time around 20h, full-time around 40h)
* `job_type` is linked to `hours_per_week`

To use your own data instead, replace the cell below with something like
`df = pd.read_csv("mydata.csv").dropna()` and list your column names in `cat_cols` and `cont_cols`.

In [ ]:
def make_synthetic_data(n=5000, seed=0):
    rng = np.random.default_rng(seed)

    education = rng.choice(["HighSchool", "Bachelor", "Master", "PhD"], size=n, p=[0.40, 0.35, 0.20, 0.05])
    area = rng.choice(["Urban", "Suburban", "Rural"], size=n, p=[0.50, 0.35, 0.15])
    age = rng.normal(40, 12, n).clip(18, 80)

    education_bonus = pd.Series(education).map(
        {"HighSchool": 0, "Bachelor": 15000, "Master": 25000, "PhD": 35000}
    ).to_numpy()
    income = 20000 + 800 * (age - 18) + education_bonus + rng.normal(0, 8000, n)

    full_time = rng.random(n) < 0.7
    hours_per_week = np.where(full_time, rng.normal(40, 3, n), rng.normal(20, 5, n))
    job_type = np.where(full_time, "full_time", "part_time")

    return pd.DataFrame({
        "education": education, "area": area, "job_type": job_type,   # categorical
        "age": age, "income": income, "hours_per_week": hours_per_week,  # continuous
    })


cat_cols = ["education", "area", "job_type"]           # M categorical variables
cont_cols = ["age", "income", "hours_per_week"]        # K continuous variables

df = make_synthetic_data()
df.head()

## 2. Split and preprocess

* We split the rows into a **training** set (the VAE learns from it) and a **validation** set
  (never used for learning; it tells us how well the model works on new data).
* The preprocessor is **fitted on the training set only**, so no information leaks from the validation set.
* Each row becomes: `[one-hot(cat_1) | ... | one-hot(cat_M) | encoded(cont_1) | ... | encoded(cont_K)]`.

In [ ]:
train_df, val_df = train_test_split(df, test_size=VAL_FRACTION, random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

# By default every continuous column uses StandardEncoder (m = 1 number per column).
# To use another encoder for a column, pass a dictionary, for example:
#     cont_encoders = {"hours_per_week": ModeSpecificEncoder()}
preprocessor = TabularPreprocessor(cat_cols, cont_cols, cont_encoders=None)

x_train = preprocessor.fit_transform(train_df)      # fit on TRAIN only
val_df = preprocessor.drop_unseen(val_df)           # (rare) categories never seen in training
x_val = preprocessor.transform(val_df)

print("Number of categories per categorical column:", preprocessor.cat_dims_)
print("Encoded width of each continuous column:    ", preprocessor.cont_dims_)
print(f"Each row has {preprocessor.input_dim_} numbers "
      f"({preprocessor.cat_total_} categorical + {preprocessor.cont_total_} continuous)")
print(f"Rows: {len(x_train)} train, {len(x_val)} validation")

# A peek at the first 3 preprocessed rows:
pd.DataFrame(x_train[:3]).round(2)

A `DataLoader` hands the data to the model in small random **batches**.

In [ ]:
train_loader = DataLoader(TensorDataset(torch.from_numpy(x_train)), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.from_numpy(x_val)), batch_size=BATCH_SIZE, shuffle=False)

## 3. The model

Open `vae.py` and read `TabularVAE`. It has an **encoder** (row → `mu`, `logvar`), a sampling step
(`z = mu + sigma * eps`), and a **decoder** (`z` → reconstructed row).

In [ ]:
model = TabularVAE(
    input_dim=preprocessor.input_dim_,
    cat_dims=preprocessor.cat_dims_,
    hidden_dim=HIDDEN_DIM,
    latent_dim=LATENT_DIM,
)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)
print("Number of trainable parameters:", sum(p.numel() for p in model.parameters()))

## 4. The loss function

`vae_loss` (in `vae.py`) adds up **three** terms, each averaged per row:

| Term | Meaning | Formula used |
|---|---|---|
| `recon_cat` | did we reconstruct the categorical variables? | cross-entropy, one per categorical variable |
| `recon_cont` | did we reconstruct the continuous variables? | squared error |
| `kl` | is the latent space close to a standard normal? | KL divergence |

```
loss = recon_cat + recon_cont + beta * kl
```

Below, `run_epoch` does one full pass through a dataset. With an optimizer it **trains**;
without one it only **evaluates** (used for the validation set).

In [ ]:
def run_epoch(model, loader, beta, optimizer=None):
    """One pass over `loader`. Returns the average per-row recon_cat, recon_cont and kl."""
    training = optimizer is not None
    model.train(training)                         # switches between training and evaluation mode

    sums = {"recon_cat": 0.0, "recon_cont": 0.0, "kl": 0.0}
    n_rows = 0

    with torch.set_grad_enabled(training):        # no gradients needed when just evaluating
        for (x,) in loader:
            x_hat, mu, logvar = model(x)                                   # forward pass
            loss, parts = vae_loss(x, x_hat, mu, logvar, preprocessor.cat_dims_, beta)

            if training:
                optimizer.zero_grad()             # 1) clear old gradients
                loss.backward()                   # 2) compute new gradients
                optimizer.step()                  # 3) update the weights

            n_rows += x.size(0)
            for name in sums:
                sums[name] += parts[name] * x.size(0)

    return {name: value / n_rows for name, value in sums.items()}

## 5. Train

Every epoch we train on the training set and then measure the loss on the validation set.

Two details:

* **KL warm-up:** at the start, the KL weight is small and grows to `BETA`. This helps the model learn
  to use `z` before the KL term starts pulling it towards N(0, 1).
* The *logged* total loss always uses the full `BETA`, so the numbers are comparable across epochs.
* We keep a copy of the model from the epoch with the **lowest validation loss**.

In [ ]:
if USE_WANDB:
    import wandb
    wandb.init(project=WANDB_PROJECT, config={
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LEARNING_RATE, "hidden_dim": HIDDEN_DIM,
        "latent_dim": LATENT_DIM, "beta": BETA, "kl_warmup_epochs": KL_WARMUP_EPOCHS,
    })


def full_loss(parts):
    return parts["recon_cat"] + parts["recon_cont"] + BETA * parts["kl"]


history = []                         # one dictionary of numbers per epoch
best_val_loss = float("inf")
best_state = None

for epoch in range(1, EPOCHS + 1):
    beta_now = BETA * min(1.0, epoch / max(1, KL_WARMUP_EPOCHS))      # KL warm-up

    train_parts = run_epoch(model, train_loader, beta_now, optimizer)   # learn
    val_parts = run_epoch(model, val_loader, beta_now)                  # just measure

    row = {"epoch": epoch, "train_loss": full_loss(train_parts), "val_loss": full_loss(val_parts)}
    for name in ["recon_cat", "recon_cont", "kl"]:
        row[f"train_{name}"] = train_parts[name]
        row[f"val_{name}"] = val_parts[name]
    history.append(row)

    if USE_WANDB:
        wandb.log(row)

    note = ""
    if row["val_loss"] < best_val_loss:
        best_val_loss = row["val_loss"]
        best_state = copy.deepcopy(model.state_dict())
        note = "  <- best so far"

    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:3d} | train loss {row['train_loss']:7.3f} | val loss {row['val_loss']:7.3f}"
              f" | val: cat {row['val_recon_cat']:5.3f}, cont {row['val_recon_cont']:5.3f}, kl {row['val_kl']:5.3f}{note}")

model.load_state_dict(best_state)    # go back to the best epoch
history = pd.DataFrame(history)
print(f"\nBest validation loss: {best_val_loss:.3f}")

if USE_WANDB:
    wandb.finish()

## 6. Read the loss curves

* **Train and validation both go down:** the model is learning.
* **Train goes down but validation goes up:** *overfitting* (the model memorizes the training rows).
* **KL close to 0:** *posterior collapse*. The decoder ignores `z`, so generated data will look generic.
  Try a smaller `BETA` or a longer `KL_WARMUP_EPOCHS`.

In [ ]:
panels = [("loss", "Total loss"),
          ("recon_cat", "Categorical reconstruction"),
          ("recon_cont", "Continuous reconstruction"),
          ("kl", "KL divergence")]

fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
for ax, (key, title) in zip(axes, panels):
    ax.plot(history["epoch"], history[f"train_{key}"], label="train")
    ax.plot(history["epoch"], history[f"val_{key}"], label="validation")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Generate synthetic data

To create new rows we do not need the encoder at all: draw `z` from a standard normal
distribution, run it through the decoder, and turn the numbers back into a table
(`preprocessor.inverse_transform`).

Then we compare with the real data. Does the synthetic table look like the real one?

In [ ]:
model.eval()
synthetic_matrix = model.sample(5000).numpy()
synthetic_df = preprocessor.inverse_transform(synthetic_matrix)
synthetic_df.head()

In [ ]:
# Continuous columns: compare basic statistics
comparison = pd.concat(
    {"real": train_df[cont_cols].describe().T[["mean", "std", "min", "max"]],
     "synthetic": synthetic_df[cont_cols].describe().T[["mean", "std", "min", "max"]]},
    axis=1,
)
display(comparison.round(2))

# Categorical columns: compare the share of each category
for c in cat_cols:
    table = pd.concat({"real": train_df[c].value_counts(normalize=True),
                       "synthetic": synthetic_df[c].value_counts(normalize=True)}, axis=1).fillna(0)
    display(table.round(3))

In [ ]:
# Did the VAE keep the relationship between job_type and hours_per_week?
print("Average hours per week by job type")
print(pd.concat({"real": train_df.groupby("job_type")["hours_per_week"].mean(),
                 "synthetic": synthetic_df.groupby("job_type")["hours_per_week"].mean()}, axis=1).round(1))

## 8. Exercises

1. Set `LATENT_DIM` to 1, 2, 8 and 32 and re-run from section 3. What happens to the reconstruction terms and the KL term?
2. Try `BETA = 0.1`, `1`, `4`. Which gives the most realistic synthetic data? Which gives the lowest reconstruction error?
3. In section 2, use `cont_encoders = {"hours_per_week": ModeSpecificEncoder()}`. Does the synthetic `hours_per_week` now show two peaks?
   (Plot a histogram of real vs synthetic to check.)
4. Write your own encoder (for example min-max scaling) with the four methods described at the top of `encoders.py`, and plug it in.
5. Set `USE_WANDB = True` and compare several runs with different settings in the Weights & Biases dashboard.
6. *Advanced:* with `ModeSpecificEncoder`, the "which peak" one-hot columns are trained with squared error.
   Change `vae_loss` so that part uses cross-entropy instead.

**Notes and limitations:** rows with missing values must be dropped or filled before preprocessing; the continuous
reconstruction assumes a Gaussian with fixed variance (that is what squared error means).